In [1]:
import json
problems = []
solutions = []
outputs = []
gts = []
file_path = "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_train/1/0/8192/predictions.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["problem"])
        solutions.append(item["solution"])
        outputs.append(item["model_generation"][0])
# Create prompt texts from problems


In [2]:
questions = []
positive_answers = []
negative_answers = []
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig
extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())

for question, output, solution in zip(problems, outputs, solutions):
    gold = parse(solution, extraction_config=extraction_target)
    answer = parse(output, extraction_config=extraction_target)
    result = verify(gold, answer)
    if not result:
        questions.append(question)
        negative_answers.append(output)
        positive_answers.append(solution)

formatted_positive = []
formatted_negative = []
for i in range(len(questions)):
    formatted_positive.append("Please reason step by step, and put your final answer within \\boxed{}.\nUser:" + questions[i] + "\nAssistant: <think>" + positive_answers[i])
    formatted_negative.append("Please reason step by step, and put your final answer within \\boxed{}.\nUser:" + questions[i] + "\nAssistant: <think>" + negative_answers[i])

In [3]:
print(len(formatted_positive))
print(len(formatted_negative))

formatted_positive = formatted_positive[:5]
formatted_negative = formatted_negative[:5]


1318
1318


In [4]:
import easysteer.hidden_states as hs
from vllm import LLM
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    task="embed", 
    tensor_parallel_size=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False
)
all_hidden_states, outputs = hs.get_all_hidden_states(llm, formatted_positive+formatted_negative)

/media/volume/llm/miniconda3/envs/easysteer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-17 16:15:36 [utils.py:253] non-default args: {'task': 'embed', 'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 11-17 16:15:36 [model.py:657] Resolved architecture: Qwen2ForCausalLM
INFO 11-17 16:15:36 [model.py:1746] Using max model len 131072


2025-11-17 16:15:37,279	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-17 16:15:37 [vllm.py:414] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:37 [core.py:94] Initializing a V1 LLM engine (v0.1.dev10891+ge8dee828a) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otl

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.88it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.87it/s]
(EngineCore_DP0 pid=3935363) 


(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:40 [default_loader.py:314] Loading weights took 0.63 seconds
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:40 [hidden_states_model_runner_mixin.py:90] Wrapped 28 decoder layers for hidden states capture
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:41 [gpu_model_runner.py:2971] Model loading took 2.9105 GiB and 1.055066 seconds
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:42 [gpu_worker.py:343] Available KV cache memory: 60.12 GiB
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:43 [kv_cache_utils.py:1247] GPU KV cache size: 2,251,472 tokens
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:43 [kv_cache_utils.py:1252] Maximum concurrency for 131,072 tokens per request: 17.18x
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:43 [core.py:238] init engine (profile, create kv cache, warmup model) took 1.84 seconds
(EngineCore_DP0 pid=3935363) INFO 11-17 16:15:43 [vllm.py:414] Cudagraph is disabled under eager mode
INFO 11-17 16:15:44 [llm.py:346] Suppor

Processed prompts: 100%|██████████| 10/10 [00:00<00:00, 21.52it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [5]:
from easysteer.steer import extract_diffmean_control_vector, StatisticalControlVector
control_vector = extract_diffmean_control_vector(
    all_hidden_states=all_hidden_states, 
    positive_indices=list(range(len(formatted_positive))),  
    negative_indices=list(range(len(formatted_positive), len(formatted_positive)+len(formatted_negative))),  
    model_type="llama",
    token_pos=-1,
    normalize=True
)
os.makedirs("/media/volume/llm/llm_steering_reasoning/vectors/basic/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/5", exist_ok=True)
control_vector.export_gguf("/media/volume/llm/llm_steering_reasoning/vectors/basic/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/5/reason.gguf")

Computing DiffMean directions: 100%|██████████| 28/28 [00:00<00:00, 7451.34it/s]


In [6]:
del llm
